# Bolånerisk-kalkylator
Interaktivt verktyg för att bedöma default-risk på bolån med hjälp av en tränad logistisk regressionsmodell (Gini: 0.426).

**Användning:** Justera sliders för att simulera olika låntagarprofiler. Modellen predikterar sannolikheten för betalningsinställelse i realtid.

| Risknivå | Default-risk | Rekommendation |
|---|---|---|
| 🟢 Låg | < 5% | Godkänn |
| 🟡 Medel | 5–20% | Manuell granskning |
| 🔴 Hög | > 20% | Avslå |

In [1]:
import pickle
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

with open('bolan_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

skuldkvot     = widgets.FloatSlider(value=4.5, min=1.0, max=10.0, step=0.1, description='Skuldkvot:')
kontantinsats = widgets.IntSlider(value=15, min=10, max=30, description='Kontantins %:')
lan_inkomst   = widgets.FloatSlider(value=3.0, min=1.0, max=10.0, step=0.1, description='Lån/inkomst:')
manadskostnad = widgets.IntSlider(value=8000, min=1000, max=30000, step=500, description='Månadskostnad:')
ranta         = widgets.FloatSlider(value=3.5, min=2.0, max=5.0, step=0.1, description='Ränta %:')
output        = widgets.Output()

def berakna(change=None):
    with output:
        clear_output(wait=True)
        features = pd.DataFrame(
            [[skuldkvot.value, kontantinsats.value, lan_inkomst.value, manadskostnad.value, ranta.value]],
            columns=['skuldkvot', 'kontantinsats_procent', 'lan_till_inkomst', 'manadskostnad', 'ranta']
        )
        features_scaled = scaler.transform(features)
        risk = model.predict_proba(features_scaled)[0][1] * 100

        if risk < 5:
            status = '🟢 Låg risk — Rekommenderas'
        elif risk < 20:
            status = '🟡 Medel risk — Manuell granskning'
        else:
            status = '🔴 Hög risk — Avslå'

        print(f'Default-risk: {risk:.1f}%')
        print(f'Modell-Gini:  0.426')
        print(f'Status:       {status}')

for w in [skuldkvot, kontantinsats, lan_inkomst, manadskostnad, ranta]:
    w.observe(berakna, names='value')

display(skuldkvot, kontantinsats, lan_inkomst, manadskostnad, ranta, output)
berakna()

FloatSlider(value=4.5, description='Skuldkvot:', max=10.0, min=1.0)

IntSlider(value=15, description='Kontantins %:', max=30, min=10)

FloatSlider(value=3.0, description='Lån/inkomst:', max=10.0, min=1.0)

IntSlider(value=8000, description='Månadskostnad:', max=30000, min=1000, step=500)

FloatSlider(value=3.5, description='Ränta %:', max=5.0, min=2.0)

Output()